# Mapping with Predefined Lists

In [46]:
import os, sys
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit
import List as mapping_list

In [47]:
import importlib
importlib.reload(mapping_list)

<module 'List' from 'd:\\VS CODE\\pulse\\mapping\\List.py'>

In [25]:
def normalize_dataframe(df, column_variants):
    variant_to_standard = {
        v.lower(): std_col
        for std_col, variants in column_variants.items()
        for v in variants
    }

    mapped_cols = {}
    new_columns = []
    for col in df.columns:
        col_lower = col.lower()
        if col_lower in variant_to_standard:
            std_col = variant_to_standard[col_lower]
            new_columns.append(std_col)
            mapped_cols[std_col] = col
        else:
            new_columns.append(col)

    for old_col, new_col in zip(df.columns, new_columns):
        df = df.withColumnRenamed(old_col, new_col)

    missing_cols = []
    for std_col in column_variants.keys():
        if std_col not in df.columns:
            df = df.withColumn(std_col, lit(None))
            missing_cols.append(std_col)

    schema_cols = list(column_variants.keys())
    extra_cols = [c for c in df.columns if c not in schema_cols]

    new_df = df.select(schema_cols)
    extra_df = df.select(schema_cols + extra_cols)

    print(f"Mapped columns: {mapped_cols}")
    print(f"Missing columns added: {missing_cols}")
    print(f"Extra columns retained: {extra_cols}")

    return new_df, extra_df, extra_cols, missing_cols, mapped_cols



In [44]:
base_dir = os.getcwd()
file_path_excel = os.path.join(base_dir, "./../faker/messy_customer_data.xlsx")
file_path_csv = os.path.join(base_dir, "./../faker/messy_customer.csv")

In [41]:
excel_df = pd.read_excel(file_path_excel, engine="openpyxl")
excel_df.to_csv(file_path_csv, index=False)
address_excel = pd.read_excel(r"D:\VS CODE\pulse\faker\messy_address_data.xlsx",engine="openpyxl")
address_excel.to_csv(r"D:\VS CODE\pulse\faker\messy_address_data.csv" ,index = False)

In [ ]:
spark = SparkSession.builder.appName("NormalizeData").getOrCreate()
customer_df = spark.read.csv(file_path_csv, header=True, inferSchema=True)
address_df = spark.read.csv(r"D:\VS CODE\pulse\faker\messy_address_data.csv", header=True, inferSchema=True)



In [45]:
customer_df.show(5)
address_df.show(5)

+-------+------------------+-----------------+------+-------------------+--------------------+--------+------------------+----------------+--------------------+--------------------+-----------------+-------------------+---------------+--------------+------+--------------+-------+
|cust_id|         full_name|customer_category|   sex|         birth_date|     account_created|  status|acquisition_source|   customer_tier|   created_timestamp|       email_address|     phone_number| last_purchase_date|total_purchases|loyalty_points|   age|lifetime_value|country|
+-------+------------------+-----------------+------+-------------------+--------------------+--------+------------------+----------------+--------------------+--------------------+-----------------+-------------------+---------------+--------------+------+--------------+-------+
|  10369|   Cynthia Gregory|              B2C|  Male|1971-10-04 00:00:00|2022-11-11 11:16:...|Inactive|          LinkedIn|      High Value|2024-12-21 08:06:.

# Implementing RapidFuzz

In [8]:
from rapidfuzz import fuzz, process


def rapidfuzz_column_mapping(df, missing_cols, extra_cols, mapped_cols, threshold=85):
    """
    Map columns using RapidFuzz's string matching algorithms
    Returns normalized dataframe, remaining missing columns, extra columns, and mapped columns
    """
    for missing_col in missing_cols[:]:
        # Use process.extractOne to find the best match
        match = process.extractOne(
            missing_col,
            extra_cols,
            scorer=fuzz.ratio,  # Can also use fuzz.WRatio or fuzz.token_sort_ratio
            score_cutoff=threshold,
        )

        if match:  # match will be (matched_string, score, index)
            best_match, score = match[0], match[1]
            print(f"Mapping: {best_match} -> {missing_col}: {score:.2f}")
            mapped_cols[missing_col] = best_match
            missing_cols.remove(missing_col)
            extra_cols.remove(best_match)

    # Rename columns based on mapping
    for new_col, old_col in mapped_cols.items():
        df = df.withColumnRenamed(old_col, new_col)

    print(f"Mapped columns: {mapped_cols}")
    print(f"Missing columns added: {missing_cols}")
    print(f"Extra columns retained: {extra_cols}")
    
    return df, missing_cols, extra_cols, mapped_cols




# Implementing NLTK

In [19]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import wordnet
from nltk.metrics.distance import edit_distance
from difflib import SequenceMatcher
import re

nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("averaged_perceptron_tagger")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\fahad\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\fahad\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\fahad\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\fahad\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\fahad\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [10]:
def preprocess_column_name(column):
    """Split camelCase and snake_case, convert to lowercase"""
    # Split by underscore and remove special characters
    words = "".join(c if c.isalnum() else " " for c in column).split()
    # Split camelCase
    result = []
    for word in words:
        result.extend(filter(None, re.split("([A-Z][a-z]*)", word)))
    return [w.lower() for w in result if w]

#### Implementing Jaccard Similarity

In [11]:
def jaccard_similarity(source_column, target_column):
    source_col = preprocess_column_name(source_column)
    target_col = preprocess_column_name(target_column)
    intersection = len(set(source_col).intersection(set(target_col)))
    union = len(set(source_col).union(set(target_col)))
    jaccard_similarity = intersection / union if union > 0 else 0
    return jaccard_similarity

#### Implementing Sequence Matcher

In [12]:
def sequence_matching(source_column, target_column):
    source_col = preprocess_column_name(source_column)
    target_col = preprocess_column_name(target_column)
    return SequenceMatcher(None, source_col, target_col).ratio()

#### Implementing Edit Distance

In [13]:
def editing_distance(source_column, target_column):
    source_col = preprocess_column_name(source_column)
    target_col = preprocess_column_name(target_column)
    max_len = max(len(source_col), len(target_col))
    return 1 - (edit_distance(source_col, target_col) / max_len)

#### Combining them all

In [14]:
def mapping_with_combination(df, missing_cols, extra_cols, mapped_cols, threshold=0.87):
    for missing_col in missing_cols[:]:
        best_match = None
        best_score = threshold
        for extra_col in extra_cols[:]:
            final = (
                0.4 * jaccard_similarity(missing_col, extra_col)
                + 0.3 * sequence_matching(missing_col, extra_col)
                + 0.3 * editing_distance(missing_col, extra_col)
            )
            print(f"{missing_col} -> {extra_col}: {final:.2f}")
            if final > best_score:
                best_score = final
                best_match = extra_col

        if best_match and best_score > threshold:
            print(f"Mapping: {best_match} -> {missing_col}: {best_score:.2f}")
            mapped_cols[missing_col] = best_match
            missing_cols.remove(missing_col)
            extra_cols.remove(best_match)

    for new_col, old_col in mapped_cols.items():
        df = df.withColumnRenamed(old_col, new_col)
    
    print(f"Mapped columns: {mapped_cols}")
    print(f"Missing columns added: {missing_cols}")
    print(f"Extra columns retained: {extra_cols}")
    
    return df, missing_cols, extra_cols, mapped_cols





#### Wordnet Mapping

In [15]:
# def preprocess_column_name(column):
#     name = re.sub("([A-Z][a-z]+)", r" \1", column)
#     name = re.sub("_", " ", name)
#     tokens = word_tokenize(name.lower())
#     return [token for token in tokens if token.isalpha()]

In [16]:
def get_wordnet_synsets(word):
    return wordnet.synsets(word)

In [17]:
def calculate_semantic_similarity(missing_col, extra_col):
    missing_tokens = preprocess_column_name(missing_col)
    extra_tokens = preprocess_column_name(extra_col)
    if not missing_tokens or not extra_tokens:
        return 0.0
    max_similarities = []
    for token1 in missing_tokens:
        synsets1 = get_wordnet_synsets(token1)
        if not synsets1:
            continue
        token_similarities = []
        for token2 in extra_tokens:
            synsets2 = get_wordnet_synsets(token2)
            if not synsets2:
                continue
            similarities = [
                s1.path_similarity(s2)
                for s1 in synsets1
                for s2 in synsets2
                if s1.path_similarity(s2) is not None
            ]
            if similarities:
                token_similarities.append(max(similarities))
        if token_similarities:
            max_similarities.append(max(token_similarities))
    return sum(max_similarities) / len(max_similarities) if max_similarities else 0.0

In [18]:
def semantic_column_mapping(df, missing_cols, extra_cols, mapped_cols, threshold=0.6):
    for missing_col in missing_cols[:]:
        best_match = None
        best_score = threshold
        for extra_col in extra_cols[:]:
            similarity = calculate_semantic_similarity(missing_col, extra_col)
            if similarity > best_score:
                best_score = similarity
                best_match = extra_col
        if best_match:
            print(f"Mapping: {best_match} -> {missing_col}: {best_score:.2f}")
            mapped_cols[missing_col] = best_match
            missing_cols.remove(missing_col)
            extra_cols.remove(best_match)
        for new_col, old_col in mapped_cols.items():
            df = df.withColumnRenamed(old_col, new_col)
    print(f"Mapped columns: {mapped_cols}")
    print(f"Missing columns added: {missing_cols}")
    print(f"Extra columns retained: {extra_cols}")

    return df, missing_cols, extra_cols, mapped_cols



# Implementing ydata-profiling

In [19]:
# from ydata_profiling import ProfileReport
# pdf = df.toPandas()

# profile = ProfileReport(pdf, title="Pandas Profiling Report", explorative=True)
# profile.to_file("pandas_profiling_report.html")
# desc = profile.description_set

# def pandas_profiling_mapping(df, missing_cols, extra_cols, mapped_cols, threshold=0.87)
#     for missing_col in missing_cols[:]:
#         best_match = None
#         best_score = 0

#         for extra_col in extra_cols[:]:
#             corr = abs(pdf[missing_col].corr(pdf[extra_col]))
#             if corr > best_score:
#                 best_score = corr
#                 best_match = extra_col

#         if best_match and best_score > threshold:
#             print(
#                 f"Data-based Mapping: {best_match} -> {missing_col} (corr={best_score:.2f})"
#             )
#             mapped_cols[missing_col] = best_match
#             pdf = pdf.rename(columns={best_match: missing_col})
#             extra_cols.remove(best_match)
#             missing_cols.remove(missing_col)
        
#     new_df = spark.createDataFrame(pdf)
#     return new_df, missing_cols, extra_cols, mapped_cols
# new_df, missing, extra_cols, mapped = pandas_profiling_mapping(
#     df, missing, extra_cols, mapped, threshold=0.87
# )

# print("\nNormalized DataFrame:")
# new_df.show(5)

# print("\nMissing columns:")
# print(missing)

# print("\nExtra columns:")
# print(extra_cols)

# print("\nMapped columns:")
# print(mapped)

# Implementing spaCy

In [20]:
import spacy

def spacy_column_mapping(
    df,
    missing_cols,
    extra_cols,
    mapped_cols,
    threshold=0.87
):
    nlp = spacy.load("en_core_web_md")

    for missing_col in missing_cols[:]:
        best_match = None
        best_score = threshold
        missing_doc = nlp(" ".join(preprocess_column_name(missing_col)))

        for extra_col in extra_cols[:]:
            extra_doc = nlp(" ".join(preprocess_column_name(extra_col)))
            similarity = missing_doc.similarity(extra_doc)

            if similarity > best_score:
                best_score = similarity
                best_match = extra_col

        if best_match:
            print(f"Mapping: {best_match} -> {missing_col}: {best_score:.2f}")
            mapped_cols[missing_col] = best_match
            missing_cols.remove(missing_col)
            extra_cols.remove(best_match)

    for new_col, old_col in mapped_cols.items():
        df = df.withColumnRenamed(old_col, new_col)
    
    print(f"Mapped columns: {mapped_cols}")
    print(f"Missing columns added: {missing_cols}")
    print(f"Extra columns retained: {extra_cols}")
    
    return df, missing_cols, extra_cols, mapped_cols



# Implementing Word2Vec

In [37]:
import numpy as np
from gensim.models import KeyedVectors
from gensim.models import Word2Vec
import re


def load_word2vec_model(df,extra_df):
    # Get all unique column names from both dataframes
    all_columns = list(set(df.columns + extra_df.columns))
    
    # Load pre-trained Word2Vec model - you can use different pre-trained models
    try:
        # Try loading Google's pre-trained model (you need to download this separately)
        model = KeyedVectors.load_word2vec_format(
            "GoogleNews-vectors-negative300.bin", binary=True
        )
    except:
        # Fallback to training a simple model on your column names
        # This is just a basic fallback - ideally you should use a pre-trained model
        sentences = [preprocess_column_name(col) for col in all_columns]
        model = Word2Vec(sentences, vector_size=100, window=5, min_count=1)
        model = model.wv
    return model


def calculate_word2vec_similarity(col1, col2, model):
    words1 = preprocess_column_name(col1)
    words2 = preprocess_column_name(col2)

    if not words1 or not words2:
        return 0.0
    vec1 = []
    vec2 = []

    for word in words1:
        try:
            vec1.append(model[word])
        except KeyError:
            continue

    for word in words2:
        try:
            vec2.append(model[word])
        except KeyError:
            continue

    if not vec1 or not vec2:
        return 0.0

    vec1_avg = np.mean(vec1, axis=0)
    vec2_avg = np.mean(vec2, axis=0)

    similarity = np.dot(vec1_avg, vec2_avg) / (
        np.linalg.norm(vec1_avg) * np.linalg.norm(vec2_avg)
    )
    return float(similarity)


def word2vec_column_mapping(df,extra_df, missing_cols, extra_cols, mapped_cols, threshold=0.87):
    model = load_word2vec_model(df,extra_df)

    for missing_col in missing_cols[:]:
        best_match = None
        best_score = threshold

        for extra_col in extra_cols[:]:
            similarity = calculate_word2vec_similarity(missing_col, extra_col, model)
            print(f"{missing_col} -> {extra_col}: {similarity:.2f}")
            if similarity > best_score:
                best_score = similarity
                best_match = extra_col

        if best_match:
            print(f"Mapping: {best_match} -> {missing_col}: {best_score:.2f}")
            mapped_cols[missing_col] = best_match
            missing_cols.remove(missing_col)
            extra_cols.remove(best_match)

        for new_col, old_col in mapped_cols.items():
            df = df.withColumnRenamed(old_col, new_col)

    print(f"Mapped columns: {mapped_cols}")
    print(f"Missing columns added: {missing_cols}")
    print(f"Extra columns retained: {extra_cols}")
    
    return df, missing_cols, extra_cols, mapped_cols



# Implementing Hugging Face RoBERTa Transformer

# Implementing GPT-OSS

new_df, missing, extra_cols, mapped = gptoss_schema_mapping(
    df, missing, extra_cols, mapped
)

print("\nNormalized DataFrame:")
new_df.show(5)

print("\nMissing columns:")
print(missing)

print("\nExtra columns:")
print(extra_cols)

print("\nMapped columns:")
print(mapped)

In [42]:
def mapping(df,column_variants):
    new_df, extra_df, extra_cols, missing_cols, mapped_cols = normalize_dataframe(
        df, column_variants
    )
    if missing_cols:
        print("\nAfter Initial Normalization:")
        print(f"Missing columns: {missing_cols}")
        print(f"Using ML Models to find missing columns:")
        new_df, missing_cols, extra_cols, mapped_cols = rapidfuzz_column_mapping(
            df, missing_cols, extra_cols, mapped_cols, threshold=85
        )

    if missing_cols:
        new_df, missing_cols, extra_cols, mapped_cols = mapping_with_combination(
            df, missing_cols, extra_cols, mapped_cols, threshold=0.87
        )

    if missing_cols:
        new_df, missing_cols, extra_cols, mapped_cols = semantic_column_mapping(
            df, missing_cols, extra_cols, mapped_cols, threshold=0.6
        )

    if missing_cols:
        new_df, missing_cols, extra_cols, mapped_cols = spacy_column_mapping(
            df, missing_cols, extra_cols, mapped_cols, threshold=0.87
        )

    if missing_cols:
        new_df, missing_cols, extra_cols, mapped_cols = word2vec_column_mapping(
            df,extra_df, missing_cols, extra_cols, mapped_cols
        )

    #if missing_cols:
    #   new_df, missing_cols, extra_cols, mapped_cols = roberta_similarity(
    #        df, missing_cols, extra_cols, mapped_cols, threshold=0.87
    #    )

    #if missing_cols:
    #    new_df, missing_cols, extra_cols, mapped_cols = gptoss_schema_mapping(
    #        df, missing_cols, extra_cols, mapped_cols
    #    )

    return new_df, extra_df, extra_cols, missing_cols, mapped_cols



In [48]:
new_df, extra_df, extra_cols, missing, mapped = mapping(
    customer_df, mapping_list.mapping_dict_customers
)
print("\nNormalized DataFrame:")
new_df.show(5)
print("\nDataFrame with Extra Columns:")
extra_df.show(5)
print("\nMissing columns:")
print(missing)
print("\nExtra columns:")
print(extra_cols)
print("\nMapped columns:")
print(mapped)

Mapped columns: {'customer_id': 'cust_id', 'customer_name': 'full_name', 'customer_type': 'customer_category', 'gender': 'sex', 'date_of_birth': 'birth_date', 'registration_date': 'account_created', 'customer_status': 'status', 'acquisition_channel': 'acquisition_source', 'customer_segment': 'customer_tier', 'created_at': 'created_timestamp'}
Missing columns added: []
Extra columns retained: ['email_address', 'phone_number', 'last_purchase_date', 'total_purchases', 'loyalty_points', 'age', 'lifetime_value', 'country']

Normalized DataFrame:
+-----------+------------------+-------------+------+-------------------+--------------------+---------------+-------------------+----------------+--------------------+
|customer_id|     customer_name|customer_type|gender|      date_of_birth|   registration_date|customer_status|acquisition_channel|customer_segment|          created_at|
+-----------+------------------+-------------+------+-------------------+--------------------+---------------+-----

In [49]:
new_df, extra_df, extra_cols, missing, mapped = mapping(
    address_df, mapping_list.mapping_dict_addresses
)
print("\nNormalized DataFrame:")
new_df.show(5)
print("\nDataFrame with Extra Columns:")
extra_df.show(5)
print("\nMissing columns:")
print(missing)
print("\nExtra columns:")
print(extra_cols)
print("\nMapped columns:")
print(mapped)

Mapped columns: {'address_id': 'addr_id', 'customer_id': 'customer_ref', 'city': 'city_name', 'country': 'country_name', 'is_default': 'default_flag', 'created_at': 'record_created'}
Missing columns added: ['address_type', 'state_province', 'postal_code']
Extra columns retained: ['address_category', 'street_line1', 'state_region', 'zip_postal', 'latitude', 'longitude', 'street_line2', 'address_verified', 'last_modified']

After Initial Normalization:
Missing columns: ['address_type', 'state_province', 'postal_code']
Using ML Models to find missing columns:
Mapped columns: {'address_id': 'addr_id', 'customer_id': 'customer_ref', 'city': 'city_name', 'country': 'country_name', 'is_default': 'default_flag', 'created_at': 'record_created'}
Missing columns added: ['address_type', 'state_province', 'postal_code']
Extra columns retained: ['address_category', 'street_line1', 'state_region', 'zip_postal', 'latitude', 'longitude', 'street_line2', 'address_verified', 'last_modified']
address_type